# Reward Models & Moral Foundations Dictionary: Experimentation Pipeline

## Colab set-up

Skip this whole section when running from a local clone — the cells detect that and no-op. On Colab they clone the repo, install requirements, and authenticate to HuggingFace.

**Before you start:**
1. `Runtime > Change runtime type > GPU`. An A100 is comfortable; a T4 works if you drop `BATCH_SIZE` to 128 in the set-up cell.
2. This repo is private, so cloning needs a GitHub token — a fine-grained PAT scoped to `puffables/rm-optpessimal-personas` with **Contents: Read-only**.
3. A HuggingFace token (read access), with the licence accepted on any gated model you plan to score.
4. Add both under the key icon ("Secrets") in the left sidebar as `GH_TOKEN` and `HF_TOKEN`, with "Notebook access" on. This keeps them out of the notebook.

**The branch must exist on the remote.** These cells check out `BRANCH` after cloning, so push it before running here — a local-only branch can't be cloned.

In [ ]:
# 1. Environment and GPU
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
print("environment:", "Colab" if IN_COLAB else "local (skipping the clone/install cells)")

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
else:
    print("no nvidia-smi — CPU or Apple MPS only. The sweep will run, but slowly; "
          "use MAX_ENTRIES in the set-up cell to keep a local test small.")

In [ ]:
# 2. Clone the repo (Colab only)
# The token is only in the URL for the clone itself; the remote is rewritten
# immediately after so it isn't left in .git/config. Safe to re-run.
REPO_PATH = "puffables/rm-optpessimal-personas"
REPO_DIR = "/content/rm-optpessimal-personas"
BRANCH = "MFD-Run"

if IN_COLAB:
    import os

    from google.colab import userdata

    GH_TOKEN = userdata.get("GH_TOKEN")
    if not os.path.exists(REPO_DIR):
        !git clone https://{GH_TOKEN}@github.com/{REPO_PATH}.git {REPO_DIR}
    %cd {REPO_DIR}
    !git remote set-url origin https://{GH_TOKEN}@github.com/{REPO_PATH}.git
    !git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}
    !git remote set-url origin https://github.com/{REPO_PATH}.git
    !git log -1 --oneline

In [ ]:
# 3. Install requirements (Colab only)
# Colab's preinstalled torch is already CUDA-enabled, so this mainly pins
# transformers and adds the smaller deps. nltk is in requirements but is only
# needed to regenerate the typed dictionary — not to run the sweep.
if IN_COLAB:
    !pip install -q -r requirements.txt

In [ ]:
# 4. HuggingFace auth, and check access before burning GPU time on a 401
if IN_COLAB:
    from google.colab import userdata
    from huggingface_hub import login

    login(token=userdata.get("HF_TOKEN"))

import yaml
from huggingface_hub import model_info

for entry in yaml.safe_load(open("config/reward_models.yaml")):
    try:
        model_info(entry["name"])
        print(f"OK   {entry['name']}")
    except Exception as exc:
        print(f"FAIL {entry['name']}: {exc}")

test

## Set-Up

Here, it should be easy to toggle: 

(1) if we're using KV-caching or the classic implementation.

(2) the dictionary we're using for our exhaustive search. (what we're cycling over as the assistant response that we'd like to score using our RM)

(3) the prompt phrasing variations we're using. (these are the templates for our user query -- 'I value [x] the most. What do you value most?', 'I think [x] is the greatest thing ever. What do you think is the greates thing ever?', etc.)

(4) the specific substitutions into the prompt phrasing (the [x] in 'I value [x] the most. What do you value most?')

### How to use the set-up block

The block below is the only place anything varies — it resolves the four toggles into a `runs` table (one row per output column) and a `dictionary` table (one row per candidate response), and the run cell just executes that plan. Nothing in the run cell needs editing to change the experiment.

**(1) Scoring path — `SCORING_MODE`.** `"kv_cache"` is the default and the one to use for real runs: it encodes the shared prompt prefix once and reuses its attention cache for every candidate, and it fixes the duplicate-BOS and decode/re-tokenize bugs documented in [`experiments/tokenization_bug_findings.md`](experiments/tokenization_bug_findings.md). `"fixed"` applies the same two fixes without caching (same cost as the original — isolates the fix from the speedup), and `"classic"` reproduces `get_reward_scores_from_response_token_ids` bugs and all, for comparison against the main-paper numbers. To compare paths, run each into its own `OUTPUT_DIR` — the checkpoint logic skips columns that already exist, so a second mode written to the same directory would be a no-op rather than a re-score.

**(2) Dictionary — `DICTIONARY_PATH`.** Defaults to `data/dictionaries/mfd2.0.dic`, the full-length MFD 2.0 (2,104 word-category pairs over 10 foundation/valence categories, 2,041 unique words after the 63 words filed under two foundations are deduplicated; the summary recommends the full-length version over the prototypicality-trimmed variants, and its validity is essentially the same). The loader also reads the other dictionaries already in that folder — `eloeverything_concepts.csv` (7,530 concepts) and `tokens_*.csv` (the full tokenizer vocabulary used by the main sweep) — so the same pipeline runs over any of the three. Use `DICTIONARY_CATEGORIES` to restrict to particular foundations and `MAX_ENTRIES` for a smoke test; note that unlike the main sweep the candidates here are **words, not single tokens**, so most are several tokens long and the two aren't directly comparable score-for-score.

**(3) Prompt phrasings — `config/mfd_prompts.yaml`.** Seven active templates, each pairing a value disclosure with a question, plus matched baselines with the disclosure sentence deleted and nothing else changed. Five ask back the same question they disclose (`value_most`, `greatest_thing`, `most_important_value`, `value_least`, `worst_thing`); two cross the valence (`greatest_ask_worst`, `worst_ask_greatest`), disclosing one pole and asking for the other, so echoing the disclosed word is no longer the agreeable answer. Each crossed template shares a `group` with the same-question template it should be compared against. Select a subset with `TEMPLATES`. Adding a framing normally means adding a template *and* its baseline — the baseline is what makes a shift interpretable — though `baseline_column` may be omitted where a matched baseline isn't wanted.

**(4) Substitutions — `config/mfd_substitutions.yaml`.** Four tiers, selected with `SUBSTITUTION_TIERS`. `primary` (10) and `extended` (21) are the MFD 2.0 seed words from Table 2 of the summary — the foundations' definitions, so they test whether the model shifts toward a foundation handed to it by name. Nine of Table 2's 40 seeds are absent: they're adjectives or verbs that don't fit the frame (`loyal`, `unnatural`), or nouns filed under a different foundation (`betrayal` is fairness.vice, not loyalty.vice). `sampled` (100) is 10 words per cell and is a **superset of both**: each cell's surviving seed words topped up with draws from that cell of the dictionary, so one selection gives the full 10-word set and the sub-groups stay distinguishable by their other tier label. The draws test whether the shift generalizes from any member of the category rather than its defining word. `control` (3) is non-moral. The default `["primary", "control"]` is the tractable full-coverage sweep; `sampled` is the expensive one (see sizing below). Tiers are a list per entry, so a seed word carries both `primary`/`extended` and `sampled`; a substitution is included if any of its tiers is selected.

The sampled words are drawn from `data/dictionaries/mfd2.0_typed.csv` — the dictionary with `word_type`, `frame_fit` and `number` labels per entry, built by `generate_typed_dictionary.py` — restricted to **singular** nouns and gerunds whose `frame_fit` is true. That's exactly the set that fits both frames: it fills a bare "I value ___ the most" without a determiner *and* agrees with "___ is the greatest thing ever". So `compassion` and `betraying` qualify; `hospital` (needs a determiner), `betray` (bare verb) and `rights` (plural, breaks the agreement) don't. loyalty.vice, MFD's smallest category, yields exactly 10 such words — which is what caps every cell at 10. Regenerate the sample with `python sample_mfd_substitutions.py --n 10 --seed 20260813`; it's seeded, so it reproduces exactly.

**Sizing a run.** Cost is `len(dictionary) x len(runs)` forward passes, and the set-up block prints both. The default is 2,041 entries x 56 columns = 114,296 sequences. Start with `MAX_ENTRIES = 50` and one template to confirm the plumbing, then clear both. `BATCH_SIZE` 384 suits a 40GB A100 — drop to 128 on a T4. KV-caching duplicates the cached prefix across the batch (`repeat_interleave`), so it doesn't necessarily buy batch-size headroom over `"fixed"` at equal length.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Set-up: resolves the four toggles into `dictionary` (candidate responses) and
# `runs` (output columns). Nothing below this cell needs editing to change the
# experiment. Runs on CPU — no model is loaded here.
# ─────────────────────────────────────────────────────────────────────────────
import re
import sys
from pathlib import Path

import pandas as pd
import yaml

# Repo root, whether this notebook is opened from the repo root locally or from
# /content/rm-optpessimal-personas on Colab.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "config").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

# ── (1) Scoring path ─────────────────────────────────────────────────────────
# "kv_cache" — cache the shared prompt prefix once per prompt; includes the
#              duplicate-BOS and decode/re-tokenize fixes. Use this for real runs.
# "fixed"    — same two fixes, no caching (full forward pass per candidate).
# "classic"  — reproduces get_reward_scores_from_response_token_ids exactly,
#              bugs included, for comparison against the main-paper numbers.
SCORING_MODE = "kv_cache"

MODEL_NAME = "Ray2333/GRM-Llama3.2-3B-rewardmodel-ft"  # must be in config/reward_models.yaml
BATCH_SIZE = 384  # 40GB A100; 128 on a T4

# One output directory per scoring mode, so the checkpoint-skip logic doesn't
# treat another mode's columns as already scored.
OUTPUT_DIR = REPO_ROOT / "data" / "mfd_reward_model_scores"
if SCORING_MODE != "kv_cache":
    OUTPUT_DIR = OUTPUT_DIR.with_name(f"{OUTPUT_DIR.name}_{SCORING_MODE}")

# ── (2) Dictionary: what gets scored as the assistant response ───────────────
DICTIONARY_PATH = REPO_ROOT / "data" / "dictionaries" / "mfd2.0.dic"
# Other dictionaries in that folder the loader also understands:
#   "mfd2.0_typed.csv"                                    (same 2,041 entries,
#       plus the word_type/frame_fit labels from generate_typed_dictionary.py;
#       those columns ride through into the scores CSV when you use this one)
#   "eloeverything_concepts.csv"                          (7,530 concepts)
#   "tokens_Ray2333--GRM-Llama3.2-3B-rewardmodel-ft.csv"  (full RM vocabulary)

DICTIONARY_CATEGORIES = None  # e.g. ["care.virtue", "care.vice"]; None = all
MAX_ENTRIES = None            # e.g. 50 for a smoke test; None = the whole dictionary

# How each entry is rendered as the assistant turn. "{entry}" is the bare word,
# which is what the one-word framings ask for. A sentence frame
# ("I value {entry} the most.") is a different experiment: it changes response
# length, and reward models are strongly length-sensitive, so don't mix frames
# within one output directory.
RESPONSE_TEMPLATE = "{entry}"

# ── (3)+(4) Prompt phrasings and their substitutions ─────────────────────────
PROMPTS_CONFIG = REPO_ROOT / "config" / "mfd_prompts.yaml"
SUBSTITUTIONS_CONFIG = REPO_ROOT / "config" / "mfd_substitutions.yaml"

TEMPLATES = None                              # e.g. ["value_most"]; None = all
# A substitution is included if it belongs to any tier listed here.
# "primary"  one MFD 2.0 seed word per foundation/valence (10)
# "extended" the other usable seed words for each cell (21 — nine of Table 2's
#            seeds are adjectives, verbs, or filed under another foundation)
# "sampled"  10 words per cell (100) — a superset of primary and extended:
#            each cell's seed words topped up with draws from that cell
# "control"  non-moral words (3)
# "sampled" multiplies the run: 100 substitutions x 7 templates is 705 columns,
# ~1.4M sequences. Pair it with a subset of TEMPLATES, or set MAX_ENTRIES,
# unless you have the GPU hours.
SUBSTITUTION_TIERS = ["primary", "control"]
SUBSTITUTIONS = None                          # e.g. ["fairness"]; None = all in the tiers
INCLUDE_BASELINES = True                      # the matched no-substitution prompts


def slugify(text):
    """Column-name slug — same convention as generate_persona_reward_model_scores.py."""
    return re.sub(r"[^a-z0-9]+", "_", text.strip().lower()).strip("_")


def load_dictionary(path):
    """Load a candidate-response list as a DataFrame [entry_id, text, categories].

    Understands the three dictionary formats in data/dictionaries/:
      *.dic  LIWC/MFD format — a `%`-delimited category header (id -> name),
             then one `word<TAB>id[<TAB>id...]` line per entry. 63 MFD 2.0 words
             appear under more than one category (e.g. `betray` is both
             fairness.vice and loyalty.vice), so entries are deduplicated by
             word and `categories` holds all of them, "|"-joined.
      *.csv  a concept/token list; the text column is `name`, `token_decoded`
             or `word`, whichever is present.
      *.txt  one entry per line.
    """
    path = Path(path)
    if path.suffix == ".dic":
        # Note: mfd2.0.dic is CR-terminated (classic Mac line endings); Python's
        # universal newlines handles that transparently.
        lines = [line.rstrip("\n") for line in path.read_text().splitlines()]
        marks = [i for i, line in enumerate(lines) if line.strip() == "%"]
        header, body = lines[marks[0] + 1:marks[1]], lines[marks[1] + 1:]
        cat_names = dict(line.split("\t", 1) for line in header if line.strip())

        entries = {}
        for line in body:
            if not line.strip():
                continue
            word, *cat_ids = line.split("\t")
            entries.setdefault(word, []).extend(cat_names[c] for c in cat_ids)
        df = pd.DataFrame({
            "text": list(entries),
            "categories": ["|".join(dict.fromkeys(c)) for c in entries.values()],
        })
    elif path.suffix == ".csv":
        raw = pd.read_csv(path)
        text_col = next(c for c in ("text", "name", "token_decoded", "word")
                        if c in raw.columns)
        df = pd.DataFrame({"text": raw[text_col].astype(str)})
        df["categories"] = raw["categories"] if "categories" in raw else ""
        # Labels from generate_typed_dictionary.py, kept if present.
        for extra in ("word_type", "frame_fit"):
            if extra in raw.columns:
                df[extra] = raw[extra]
        df = df.drop_duplicates("text")
    else:
        df = pd.DataFrame({"text": path.read_text().split("\n"), "categories": ""})
        df = df[df["text"].str.strip() != ""].drop_duplicates("text")

    df = df.reset_index(drop=True)
    df.insert(0, "entry_id", df.index)
    return df


# ── Resolve the dictionary ───────────────────────────────────────────────────
dictionary = load_dictionary(DICTIONARY_PATH)
if DICTIONARY_CATEGORIES:
    keep = set(DICTIONARY_CATEGORIES)
    mask = dictionary["categories"].apply(lambda c: bool(keep & set(c.split("|"))))
    dictionary = dictionary[mask].reset_index(drop=True)
if MAX_ENTRIES:
    dictionary = dictionary.head(MAX_ENTRIES).reset_index(drop=True)

dictionary["response"] = dictionary["text"].map(lambda t: RESPONSE_TEMPLATE.format(entry=t))

# Identity columns copied into the scores CSV ahead of the per-prompt columns.
ID_COLUMNS = [c for c in ("entry_id", "text", "categories", "word_type", "frame_fit")
              if c in dictionary.columns]

# ── Resolve the prompts ──────────────────────────────────────────────────────
prompts_cfg = yaml.safe_load(PROMPTS_CONFIG.read_text())
templates = prompts_cfg["templates"]
baselines = prompts_cfg.get("baselines", {})
if TEMPLATES:
    templates = {k: v for k, v in templates.items() if k in TEMPLATES}
    baselines = {k: v for k, v in baselines.items()
                 if k in {t.get("baseline_column") for t in templates.values()}}

substitutions = yaml.safe_load(SUBSTITUTIONS_CONFIG.read_text())["substitutions"]
if SUBSTITUTION_TIERS:
    wanted = set(SUBSTITUTION_TIERS)
    substitutions = [s for s in substitutions if wanted & set(s["tiers"])]
if SUBSTITUTIONS:
    keep = set(SUBSTITUTIONS)
    substitutions = [s for s in substitutions if s["name"] in keep or slugify(s["name"]) in keep]

# One row per output column, carrying each prompt's substitution metadata.
# Saved alongside the scores so analysis never has to re-derive which prompt
# produced a column, or which word it primed with.
runs = []
if INCLUDE_BASELINES:
    for name, text in baselines.items():
        runs.append({"column": name, "prompt": text, "template": None,
                     "substitution": None, "foundation": None, "valence": None,
                     "tier": "baseline", "word_type": None, "baseline_column": None,
                     "group": None})
for template_name, template in templates.items():
    for sub in substitutions:
        runs.append({
            "column": f"{template_name}__{slugify(sub['name'])}",
            "prompt": template["text"].format(substitution=sub["name"]),
            "template": template_name,
            "substitution": sub["name"],
            "foundation": sub["foundation"],
            "valence": sub["valence"],
            "tier": "|".join(sub["tiers"]),
            "word_type": sub["word_type"],
            # Optional: a template may deliberately have no matched baseline.
            "baseline_column": template.get("baseline_column"),
            "group": template.get("group"),
        })
runs = pd.DataFrame(runs)

# ── Plan summary ─────────────────────────────────────────────────────────────
n_passes = len(dictionary) * len(runs)
print(f"model        {MODEL_NAME}")
print(f"scoring      {SCORING_MODE}  (batch size {BATCH_SIZE})")
print(f"dictionary   {DICTIONARY_PATH.name}: {len(dictionary)} entries"
      f"{f', categories {DICTIONARY_CATEGORIES}' if DICTIONARY_CATEGORIES else ''}")
print(f"prompts      {len(templates)} templates x {len(substitutions)} substitutions"
      f" + {len(baselines) if INCLUDE_BASELINES else 0} baselines = {len(runs)} columns")
print(f"cost         {len(dictionary)} x {len(runs)} = {n_passes:,} scored sequences")
print(f"output       {OUTPUT_DIR.relative_to(REPO_ROOT)}/{MODEL_NAME.replace('/', '--')}.csv")
print()
print("categories:", dict(dictionary["categories"].str.split("|").explode().value_counts()))
print()
print("first 3 prompts:")
for _, r in runs.head(3).iterrows():
    print(f"  {r['column']:38s} \"{r['prompt']}\"")
print(f"first 5 candidate responses: {dictionary['response'].head(5).tolist()}")

### Persisting results across a disconnect

The sweep checkpoints after every column, but on Colab those checkpoints live in `/content`, which is wiped when the runtime is recycled — so they survive a crash mid-session and *not* the VM going away. Mounting Drive puts every checkpoint somewhere that outlives the session, which makes a disconnect cost at most the column in flight.

Run this **after** the set-up cell (it overrides `OUTPUT_DIR`, which set-up defines) and **before** the sweep. Re-running set-up afterwards resets the path, so if you change `SCORING_MODE` later, re-run set-up and then this cell again.

In [ ]:
# Optional: write results to Google Drive rather than the Colab VM's disk.
PERSIST_TO_DRIVE = False
DRIVE_SUBDIR = "mfd_experiment"

if PERSIST_TO_DRIVE and IN_COLAB:
    import shutil

    from google.colab import drive

    drive.mount("/content/drive")
    local_dir = OUTPUT_DIR
    OUTPUT_DIR = Path("/content/drive/MyDrive") / DRIVE_SUBDIR / OUTPUT_DIR.name
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Carry over any checkpoint already written on this VM, so a run that
    # started on local disk resumes here instead of re-scoring from scratch.
    for path in sorted(local_dir.glob("*.csv")) if local_dir.exists() else []:
        if not (OUTPUT_DIR / path.name).exists():
            shutil.copy2(path, OUTPUT_DIR / path.name)
            print(f"carried over checkpoint: {path.name}")

print("results will be written to:", OUTPUT_DIR)

## Experiment Run Pipeline

Run an exhaustive token search as in colab_run_pipeline, but instead using the set-up described in the Set-Up section.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Run: score every dictionary entry as the assistant response, for every prompt
# in `runs`. Checkpoints after each column, so re-running resumes rather than
# restarting. Needs a GPU.
# ─────────────────────────────────────────────────────────────────────────────
from collections import defaultdict

import torch
from tqdm.auto import tqdm
from transformers.cache_utils import DynamicCache

from reward_model_support import RewardModel
from reward_model_registry import *  # registers each model's score extraction

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / f"{MODEL_NAME.replace('/', '--')}.csv"
runs_path = output_path.with_name(f"{output_path.stem}__runs.csv")

# ── Resume ───────────────────────────────────────────────────────────────────
if output_path.exists():
    scores_df = pd.read_csv(output_path)
    # Scores are positional, so a checkpoint written against a different
    # dictionary (or a different MAX_ENTRIES/RESPONSE_TEMPLATE) can't be
    # extended — its columns would silently misalign with these rows.
    if scores_df["text"].tolist() != dictionary["text"].tolist():
        raise ValueError(
            f"{output_path} was written for a different entry set "
            f"({len(scores_df)} rows vs {len(dictionary)} here). Point OUTPUT_DIR "
            "somewhere new, or restore the set-up toggles that produced it."
        )
else:
    scores_df = dictionary[ID_COLUMNS].copy()

pending = runs[~runs["column"].isin(scores_df.columns)]
print(f"{len(pending)}/{len(runs)} columns to score "
      f"({len(runs) - len(pending)} already in {output_path.name})")

# ── Scoring paths ────────────────────────────────────────────────────────────
def _cache_layers(cache):
    """Per-layer (keys, values) of a DynamicCache, across the transformers 4.56
    rename of .key_cache/.value_cache to .layers[i].keys/.values."""
    if hasattr(cache, "key_cache"):
        return list(zip(cache.key_cache, cache.value_cache))
    return [(layer.keys, layer.values) for layer in cache.layers]


def score_classic(rm, prompt, responses, batch_size):
    """The original path: each response goes in as chat-template *text* and the
    whole conversation is re-tokenized (double BOS, decode/re-tokenize seam —
    see experiments/tokenization_bug_findings.md). Kept bug-for-bug identical to
    RewardModel.get_reward_scores_from_response_token_ids so this notebook's
    numbers are comparable to the main sweep's.
    """
    scores = []
    for i in tqdm(range(0, len(responses), batch_size), leave=False):
        conversations = [
            [{"role": "user", "content": prompt}, {"role": "assistant", "content": r}]
            for r in responses[i:i + batch_size]
        ]
        formatted = [rm.tokenizer.apply_chat_template(c, tokenize=False)
                     for c in conversations]
        scores.extend(rm._calculate_batch_scores(formatted))
    return scores


def score_prefix_shared(rm, prompt, responses, batch_size, use_cache):
    """The "fixed" (use_cache=False) and "kv_cache" (True) paths, generalized
    from RewardModel.get_reward_scores_from_response_token_ids_{fixed,kv_cached}
    to multi-token responses — MFD entries are words, not single tokens, so a
    batch is only rectangular if its responses tokenize to the same length.
    Rather than pad (which would put the sequence-classification head's
    last-token lookup at the mercy of the padding side), responses are grouped
    by token length and each group is batched separately. Same number of forward
    passes, no padding anywhere.
    """
    device = rm.device
    prefix_ids, suffix_ids = rm._build_prefix_suffix_ids(prompt)
    prefix_ids, suffix_ids = prefix_ids.to(device), suffix_ids.to(device)
    prefix_len = prefix_ids.shape[1]

    if use_cache:
        with torch.no_grad():
            prefix_out = rm.model(
                input_ids=prefix_ids,
                attention_mask=torch.ones_like(prefix_ids),
                past_key_values=DynamicCache(),
                use_cache=True,
            )
        prefix_cache = prefix_out.past_key_values

    encoded = rm.tokenizer(list(responses), add_special_tokens=False)["input_ids"]
    by_length = defaultdict(list)
    for idx, ids in enumerate(encoded):
        by_length[len(ids)].append(idx)

    scores = [float("nan")] * len(responses)
    batches = [(length, group[i:i + batch_size])
               for length, group in sorted(by_length.items())
               for i in range(0, len(group), batch_size)]

    for length, group in tqdm(batches, leave=False):
        if length == 0:  # response tokenized to nothing (e.g. a blank entry)
            continue
        n = len(group)
        response_ids = torch.tensor([encoded[j] for j in group], device=device)

        if use_cache:
            new_ids = torch.cat([response_ids, suffix_ids.expand(n, -1)], dim=1)
            new_len = new_ids.shape[1]

            # The prefix was encoded once with batch dimension 1; broadcast it
            # across this batch. Built through update() rather than by assigning
            # cache.key_cache/.value_cache (as the persona sweep in
            # reward_model_support.py does) because those attributes don't exist
            # in transformers >= 4.56 — update() works on both sides of that
            # rename, and keeps the cache's own length bookkeeping correct.
            expanded_cache = DynamicCache()
            for layer_idx, (keys, values) in enumerate(_cache_layers(prefix_cache)):
                expanded_cache.update(keys.repeat_interleave(n, dim=0),
                                      values.repeat_interleave(n, dim=0),
                                      layer_idx)

            attention_mask = torch.ones(n, prefix_len + new_len, device=device)
            position_ids = torch.arange(prefix_len, prefix_len + new_len, device=device)
            position_ids = position_ids.unsqueeze(0).expand(n, -1)

            with torch.no_grad():
                outputs = rm.model(
                    input_ids=new_ids,
                    attention_mask=attention_mask,
                    past_key_values=expanded_cache,
                    position_ids=position_ids,
                    use_cache=False,
                )
        else:
            full_ids = torch.cat([
                prefix_ids.expand(n, -1),
                response_ids,
                suffix_ids.expand(n, -1),
            ], dim=1)
            with torch.no_grad():
                outputs = rm.model(input_ids=full_ids,
                                   attention_mask=torch.ones_like(full_ids))

        for idx, score in zip(group, rm._extract_scores_from_outputs(outputs)):
            scores[idx] = score
    return scores


# ── Sweep ────────────────────────────────────────────────────────────────────
if len(pending):
    reward_model = RewardModel.create(MODEL_NAME)
    if SCORING_MODE == "kv_cache" and reward_model.multi_gpu:
        raise ValueError(f"{MODEL_NAME} is multi_gpu: true — KV-cached scoring is "
                         'single-device only. Use SCORING_MODE = "fixed".')

    score_fns = {
        "classic": lambda p, r: score_classic(reward_model, p, r, BATCH_SIZE),
        "fixed": lambda p, r: score_prefix_shared(reward_model, p, r, BATCH_SIZE, False),
        "kv_cache": lambda p, r: score_prefix_shared(reward_model, p, r, BATCH_SIZE, True),
    }
    score_fn = score_fns[SCORING_MODE]
    responses = dictionary["response"].tolist()

    for _, run in tqdm(list(pending.iterrows()), desc="prompts"):
        print(f'{run["column"]}: "{run["prompt"]}"')
        scores_df[run["column"]] = score_fn(run["prompt"], responses)
        scores_df.to_csv(output_path, index=False, escapechar="\\")  # checkpoint

    del reward_model
    torch.cuda.empty_cache()

# Column -> prompt provenance, merged so resumed/partial runs keep earlier rows.
if runs_path.exists():
    previous = pd.read_csv(runs_path)
    runs_out = pd.concat([previous[~previous["column"].isin(runs["column"])], runs])
else:
    runs_out = runs
runs_out.to_csv(runs_path, index=False)

print(f"\nSaved {output_path} — {len(scores_df)} entries x "
      f"{len(scores_df.columns) - len(ID_COLUMNS)} prompt columns")
scores_df.head()

## Analysis/Figures

### kv-cache vs no-cache

Compares the scoring paths against each other, for whichever of the three output directories exist. Run the same sweep under two modes first: set `SCORING_MODE`, **re-run the set-up cell** (that's where `OUTPUT_DIR` is derived — changing the mode without re-running it would write the second mode's columns into the first mode's file), then re-run the sweep.

What to expect: `fixed` vs `kv_cache` isolates caching alone and should agree to floating-point noise — a mean absolute difference on the order of 1e-3 in fp16 (0.003 with max 0.008 on a 12-word pilot), with identical top-10s. A real gap there is a caching bug, not a finding. `classic` vs either also carries the duplicate-BOS and decode/re-tokenize bugs, so it diverges much further; that difference is the bug's effect, and it's the reason the main sweep's numbers aren't directly comparable to these.

In [ ]:
# Requires the set-up cell to have run (MODEL_NAME, REPO_ROOT, ID_COLUMNS).
from itertools import combinations

import pandas as pd

MODEL_CSV = f"{MODEL_NAME.replace('/', '--')}.csv"
SCORES_ROOT = REPO_ROOT / "data"

paths = {
    "kv_cache": SCORES_ROOT / "mfd_reward_model_scores" / MODEL_CSV,
    "fixed": SCORES_ROOT / "mfd_reward_model_scores_fixed" / MODEL_CSV,
    "classic": SCORES_ROOT / "mfd_reward_model_scores_classic" / MODEL_CSV,
}
variants = {mode: pd.read_csv(path) for mode, path in paths.items() if path.exists()}
print("found:", ", ".join(variants) or "nothing — run a sweep first")

for a, b in combinations(variants, 2):
    left, right = variants[a], variants[b]
    if left["text"].tolist() != right["text"].tolist():
        print(f"\n{a} vs {b}: different entry sets, not comparable")
        continue

    shared = [c for c in left.columns
              if c in right.columns and c not in ID_COLUMNS]
    if not shared:
        print(f"\n{a} vs {b}: no prompt columns in common")
        continue

    rows = []
    for col in shared:
        diff = (left[col] - right[col]).abs()
        top_a = set(left.nlargest(10, col)["text"])
        top_b = set(right.nlargest(10, col)["text"])
        rows.append({"column": col, "mean|diff|": diff.mean(), "max|diff|": diff.max(),
                     "top10_overlap": len(top_a & top_b)})
    summary = pd.DataFrame(rows).set_index("column")
    print(f"\n=== {a} vs {b} ({len(shared)} columns) ===")
    print(f"  mean|diff| {summary['mean|diff|'].mean():.4f} "
          f"(worst column {summary['mean|diff|'].max():.4f}), "
          f"max|diff| {summary['max|diff|'].max():.4f}, "
          f"top-10 overlap {summary['top10_overlap'].mean():.1f}/10")
    display(summary.sort_values("mean|diff|", ascending=False).head(10))

### Median rank by dictionary category

For each prompt, rank all dictionary entries by score (rank 1 = the entry the reward model liked most) and take the median rank of each of the ten foundation/valence categories. Each category's median is then compared against the same category's median under that prompt's **matched baseline**, because categories differ a lot in their unprompted standing — the model has opinions about care.virtue words whatever you ask it — so a raw median mostly measures that, while the delta isolates what the disclosure did. Negative delta = the category moved up the ranking.

Three relations, defined against the category the prompt primed:

| relation | meaning |
|---|---|
| within category - same pole | primed care.virtue, ranking care.virtue |
| within category - opposite pole | primed care.virtue, ranking care.vice |
| across category | primed care.virtue, ranking the eight categories of other foundations |

Two caveats worth carrying into the reading. The echoed substitution stays in its own category — it's one entry among dozens so it barely moves a median, but it is not removed. And ranks are zero-sum: if one category rises, the rest must fall, so the interesting comparison is always same-pole against the other two relations rather than any one number's sign.

In [ ]:
# ── Median rank by dictionary category ───────────────────────────────────────
# For every prompt column, rank all dictionary entries by score (rank 1 = the
# entry the reward model scored highest), then take the median rank of each of
# the 10 foundation/valence categories. The comparison of interest is against
# the same category's median under the prompt's matched baseline: categories
# differ a lot in their unprompted standing (the model likes care.virtue words
# whatever you ask it), so a raw median mostly measures that, while the delta
# isolates what the disclosure did. Negative delta = the category moved up.
#
# Entries filed under two foundations count once in each. The echoed
# substitution stays in its category, as it's one entry among dozens.
import numpy as np
import pandas as pd

ANALYSIS_DIR = REPO_ROOT / "mfd_analysis" / "output"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

scores_path = OUTPUT_DIR / f"{MODEL_NAME.replace('/', '--')}.csv"
runs_path = scores_path.with_name(f"{scores_path.stem}__runs.csv")
scores = pd.read_csv(scores_path)
runs_meta = pd.read_csv(runs_path)

prompt_cols = [c for c in scores.columns if c not in ID_COLUMNS]
n_entries = len(scores)

# rank within each column, then one row per (column, category) pair
ranks = scores[prompt_cols].rank(ascending=False, method="average")
ranks["categories"] = scores["categories"]
long = ranks.melt(id_vars="categories", var_name="column", value_name="rank")
long["category"] = long["categories"].str.split("|")
long = long.explode("category").dropna(subset=["category", "rank"])

median_rank = (long.groupby(["column", "category"])["rank"]
               .median().reset_index(name="median_rank"))

# attach each column's prompt metadata, then its baseline's median for the
# same category
by_category = median_rank.merge(runs_meta, on="column", how="left")
baselines = median_rank.rename(columns={"column": "baseline_column",
                                        "median_rank": "baseline_median_rank"})
by_category = by_category.merge(baselines, on=["baseline_column", "category"], how="left")
by_category["delta"] = by_category["median_rank"] - by_category["baseline_median_rank"]
# percentile is comparable across dictionaries of different size
by_category["median_percentile"] = 100 * by_category["median_rank"] / n_entries


def relation(row):
    """How a dictionary category relates to the category the prompt primed."""
    if pd.isna(row["foundation"]) or row["foundation"] == "none":
        return "baseline" if row["tier"] == "baseline" else "control prompt"
    if row["category"] == f"{row['foundation']}.{row['valence']}":
        return "within category - same pole"
    if row["category"].split(".")[0] == row["foundation"]:
        return "within category - opposite pole"
    return "across category"


by_category["relation"] = by_category.apply(relation, axis=1)
by_category.to_csv(ANALYSIS_DIR / "median_rank_by_category.csv", index=False)
print(f"{len(by_category)} (column x category) rows -> "
      f"{ANALYSIS_DIR / 'median_rank_by_category.csv'}")
print(by_category["relation"].value_counts().to_string())

In [ ]:
# The 10x10 picture: what each primed cell does to each dictionary category.
# Rows are the foundation/valence the prompt primed, columns are the category
# being ranked, cells are the mean delta in median rank across every
# substitution in that row (and every selected template). The diagonal is
# "within category - same pole".
TEMPLATE_FILTER = None  # e.g. ["greatest_thing"]; None = pool all templates

primed = by_category[by_category["relation"] != "baseline"].copy()
if TEMPLATE_FILTER:
    primed = primed[primed["template"].isin(TEMPLATE_FILTER)]
primed["primed_cell"] = primed["foundation"] + "." + primed["valence"]

CELL_ORDER = [f"{f}.{v}" for f in ("care", "fairness", "loyalty", "authority", "sanctity")
              for v in ("virtue", "vice")]

matrix = (primed.pivot_table(index="primed_cell", columns="category",
                             values="delta", aggfunc="mean")
          .reindex(index=CELL_ORDER, columns=CELL_ORDER))

print(f"mean delta in median rank, {len(primed['column'].unique())} prompt columns "
      f"({'all templates' if not TEMPLATE_FILTER else ', '.join(TEMPLATE_FILTER)})")
print("negative = category ranked better than under the matched baseline\n")
display(matrix.style
        .background_gradient(cmap="RdBu", axis=None)
        .format("{:+.0f}")
        .set_caption("rows: primed foundation.valence &nbsp;|&nbsp; columns: dictionary category"))

In [ ]:
# Summary by relation, in the style of personaFigures/gen_figures.ipynb
# FIGURE 1: one panel per template, individual runs as faint jittered points
# with the mean +- s.d. overlaid.
#
# One point is one substitution run. Within a run, a relation that spans
# several categories (across category covers the eight belonging to other
# foundations) is averaged to a single value first, so every point carries the
# same weight regardless of how many categories it summarizes.
import matplotlib.pyplot as plt

RELATIONS = ["within category - same pole",
             "within category - opposite pole",
             "across category"]
GRID_ALPHA, POINT_ALPHA, MEAN_SD_ALPHA = 0.15, 0.35, 0.75
REL_COLOR = {RELATIONS[0]: "#1b6ca8", RELATIONS[1]: "#c1553b", RELATIONS[2]: "#5a5a5a"}

per_run = (primed[primed["relation"].isin(RELATIONS)]
           .groupby(["template", "column", "substitution", "relation"], as_index=False)["delta"]
           .mean())

templates_present = [t for t in runs_meta["template"].dropna().unique()
                     if t in set(per_run["template"])]
rng = np.random.default_rng(0)

ncols = min(4, len(templates_present))
nrows = -(-len(templates_present) // ncols)  # ceiling division
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 4.6 * nrows),
                         sharey=True, squeeze=False)
flat = axes.flatten()
for ax in flat[len(templates_present):]:
    ax.set_visible(False)
for ax, template in zip(flat, templates_present):
    sub = per_run[per_run["template"] == template]
    ax.axhline(0, color="black", linewidth=0.8, alpha=0.6)
    ax.set_axisbelow(True)
    ax.grid(True, alpha=GRID_ALPHA, linewidth=0.6)
    ax.set_xticks(range(len(RELATIONS)))
    ax.set_xticklabels([r.replace("within category - ", "within,\n").replace("across category", "across\ncategory")
                        for r in RELATIONS])
    ax.set_title(template, fontsize=10)

    stats = sub.groupby("relation")["delta"].agg(["mean", "std"]).reindex(RELATIONS)
    for i, rel in enumerate(RELATIONS):
        vals = sub.loc[sub["relation"] == rel, "delta"].to_numpy()
        ax.scatter(np.full(len(vals), i) + rng.uniform(-0.15, 0.15, len(vals)), vals,
                   s=15, alpha=POINT_ALPHA, color=REL_COLOR[rel], linewidths=0, zorder=1)
    ax.errorbar(range(len(RELATIONS)), stats["mean"], yerr=stats["std"], fmt="o",
                color="black", ecolor="black", elinewidth=1.5, capsize=4,
                markersize=6, zorder=2, alpha=MEAN_SD_ALPHA)

for row in range(nrows):
    axes[row][0].set_ylabel("change in median rank vs. matched baseline\n(negative = ranked better)")
fig.suptitle("Median rank shift by relation to the primed category "
             f"({MODEL_NAME.split('/')[-1]}, {SCORING_MODE})")
fig.tight_layout()
fig.savefig(ANALYSIS_DIR / "median_rank_by_relation.png", dpi=300, bbox_inches="tight")
plt.show()

summary = (per_run.groupby("relation")["delta"].agg(["mean", "std", "count"])
           .reindex(RELATIONS).round(1))
print("pooled across templates (average and s.d. of per-run medians):")
print(summary.to_string())

TODO: TBD

## Pull results back out

Colab sessions are ephemeral, so download anything new before the runtime disconnects. `data/` is checked into git, so `git status` in your local clone will show only what actually changed.

In [ ]:
if IN_COLAB:
    from google.colab import files

    !zip -qr mfd_outputs.zip data/mfd_reward_model_scores*
    files.download("mfd_outputs.zip")
else:
    print("local run — results are already in", OUTPUT_DIR)